In [ ]:
### Script for plotting the SED_PPE hazard curves
import h5py as h5
import numpy as np
import os
import matplotlib.pyplot as plt
import geopandas as gpd
from glob import glob

In [ ]:
results_dir = "../results/CFM/sites_c_MDEz/N34_b0959_C42_S10_TI_geologic_c_MDEz_sites"
results_dirs = glob("../results/CFM/*/*_sites")
plots_dir = os.path.join(os.path.dirname(os.path.dirname(results_dir)), "SED_PPE_plots")
os.makedirs(plots_dir, exist_ok=True)

site_geojson = "../crustal/discretised_CFM/crustal_site_locations_10km_coast.geojson"

years = 7500

sites = gpd.read_file(site_geojson)
sites

In [ ]:
for siteId in sites.siteId:
    plt.close()
    if os.path.exists(os.path.join(plots_dir, f"{siteId}_SED.png")):
        continue
    plot_site = True
    print(siteId)
    for results_dir in results_dirs:
        # print(results_dir)
        if os.path.exists(f"{results_dir}/{siteId}.h5"):
            delete_site = False
            with h5.File(f"{results_dir}/{siteId}.h5", 'r') as site_h5:
                if str(years) not in site_h5:
                    plot_site = False
                    continue
                if "SED_PPE" not in site_h5[str(years)]:
                    plot_site = False
                    continue
                try:
                    SED_THRESH = np.arange(site_h5[str(years)]["SED_PPE"]["SED_THRESH"][0], site_h5[str(years)]["SED_PPE"]["SED_THRESH"][1] + site_h5[str(years)]["SED_PPE"]["SED_THRESH"][2], site_h5[str(years)]["SED_PPE"]["SED_THRESH"][2])
                except KeyError:
                    delete_site = True
                if plot_site:
                    up_SED = site_h5[str(years)]['SED_PPE']['up_SED_PPE'][:]
                    down_SED = site_h5[str(years)]['SED_PPE']['down_SED_PPE'][:]
                    try:
                        plt.plot(SED_THRESH[:up_SED.size], up_SED * 100, 'r')
                        plt.plot(SED_THRESH[:down_SED.size], down_SED * 100, 'b')
                    except ValueError:
                        delete_site = True

            if delete_site:
                print(f"[DELETING]: {results_dir}/{siteId}.h5")
                os.remove(f"{results_dir}/{siteId}.h5")
    if plot_site:
        plt.xlabel("PPE threshold (m)")
        plt.ylabel("Probability of Exceedance (%)")
        plt.xscale('log')
        plt.yscale('log')
        plt.ylim([0.1, 110])
        plt.xlim([0.1, 30])
        plt.title(f'Site: {siteId}, {years} yrs with 1,000,000 samples')
        plt.savefig(os.path.join(plots_dir, f"{siteId}_SED.png"))


